# Modelo Predictivo — Demanda Horaria ZMVM

Se comparan dos modelos para predecir la demanda eléctrica horaria del área CEN (ZMVM), entrenados por separado para cada periodo del quiebre estructural 2020:

- **Random Forest** — modelo principal, captura relaciones no lineales entre clima, calendario y demanda. Aporta feature importances interpretables.
- **Prophet** — modelo explicativo, descompone la serie en tendencia + estacionalidad diaria + semanal + anual. Incorpora temperatura como regresor externo.

Los modelos se evalúan con MAE, RMSE y MAPE sobre un conjunto de test temporal (últimos 20% de cada periodo).

### Carga de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")

PROCESSED_DIR = Path("..") / "data" / "processed"

## 1. Carga y preparación de datos

In [ ]:
df_a = pd.read_parquet(PROCESSED_DIR / "features_2016_2019.parquet")
df_b = pd.read_parquet(PROCESSED_DIR / "features_2020_2024.parquet")

# Features para el modelo horario — se excluyen indicadores económicos (baja señal a nivel horario)
# y variables categóricas no codificadas
FEATURES = [
    "hora", "dia_semana", "mes",
    "es_festivo", "es_fin_semana", "dia_no_laboral",
    "temperatura", "humedad", "radiacion",
    "temperatura_lag1h", "temperatura_lag2h",
    "demanda_lag_1h", "demanda_lag_24h", "demanda_lag_168h",
]
TARGET = "demanda_balance_mwh"

print("Periodo A:", df_a.shape, "| Periodo B:", df_b.shape)
print("Features:", len(FEATURES))

## 2. Split temporal train/test

Se toman los últimos 20% de registros de cada periodo como conjunto de test, respetando el orden temporal. **No se usa shuffle** — mezclar el tiempo invalidaría los lags de demanda como features.

In [ ]:
def temporal_split(df, features, target, test_frac=0.2):
    n = len(df)
    n_test = int(n * test_frac)
    train = df.iloc[:-n_test]
    test  = df.iloc[-n_test:]
    return (
        train[features], train[target],
        test[features],  test[target],
    )

X_train_a, y_train_a, X_test_a, y_test_a = temporal_split(df_a, FEATURES, TARGET)
X_train_b, y_train_b, X_test_b, y_test_b = temporal_split(df_b, FEATURES, TARGET)

print("Periodo A — train:", len(X_train_a), "| test:", len(X_test_a))
print("Periodo B — train:", len(X_train_b), "| test:", len(X_test_b))

## 3. Random Forest

Se entrena un `RandomForestRegressor` por separado para cada periodo. Hiperparámetros conservadores para un primer modelo: 300 árboles, profundidad máxima 20, mínimo 10 muestras por hoja. El número de features por árbol (`max_features="sqrt"`) reduce la correlación entre árboles y mejora la generalización.

In [ ]:
RF_PARAMS = dict(n_estimators=300, max_depth=20, min_samples_leaf=10,
                 max_features="sqrt", n_jobs=-1, random_state=42)

rf_a = RandomForestRegressor(**RF_PARAMS)
rf_a.fit(X_train_a, y_train_a)
pred_rf_a = rf_a.predict(X_test_a)

rf_b = RandomForestRegressor(**RF_PARAMS)
rf_b.fit(X_train_b, y_train_b)
pred_rf_b = rf_b.predict(X_test_b)

def metrics(y_true, y_pred, label=""):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{label:30s}  MAE={mae:7.1f} MWh  RMSE={rmse:7.1f} MWh  MAPE={mape:.2f}%")
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}

print("=== Random Forest ===")
m_rf_a = metrics(y_test_a, pred_rf_a, "RF — Periodo A (2016-2019)")
m_rf_b = metrics(y_test_b, pred_rf_b, "RF — Periodo B (2020-2024)")

### 3.1 Feature importances

Las importancias de Random Forest (impureza media de Gini, promediada sobre los árboles) permiten verificar que los lags de demanda dominan el modelo y que las variables climáticas aportan señal secundaria, como se esperaba del análisis de correlación.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, rf, title in [
    (axes[0], rf_a, "Feature Importances — Periodo A (2016-2019)"),
    (axes[1], rf_b, "Feature Importances — Periodo B (2020-2024)"),
]:
    imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
    colors = ["#e07b54" if "lag" in i else "#5a8fa8" if "temperatura" in i or "humedad" in i or "radiacion" in i
              else "#7ab87a" for i in imp.index]
    imp.plot(kind="barh", ax=ax, color=colors)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Importancia (Gini)")
    ax.axvline(0, color="gray", linewidth=0.5)

plt.tight_layout()
plt.savefig("../docs/img/rf_feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Prophet

Prophet descompone la serie en tendencia + estacionalidades múltiples (diaria, semanal, anual). Se agrega `temperatura` como regresor externo para incorporar el efecto climático. Por limitaciones de memoria y velocidad, Prophet se entrena con una **muestra representativa** de 15,000 puntos de cada periodo (muestreo temporal, preservando estructura).

In [ ]:
def prep_prophet(df, target=TARGET, max_rows=15_000):
    """Transforma el dataframe al formato ds/y que requiere Prophet."""
    sub = df[[target, "temperatura"]].copy()
    sub.index = pd.to_datetime(sub.index)
    sub = sub.sort_index()
    if len(sub) > max_rows:
        # muestra temporal uniforme — no shuffle
        idx = np.linspace(0, len(sub) - 1, max_rows, dtype=int)
        sub = sub.iloc[idx]
    sub = sub.rename(columns={target: "y"})
    sub = sub.reset_index().rename(columns={"datetime": "ds", sub.index.name: "ds"})
    if "ds" not in sub.columns:
        sub.insert(0, "ds", sub.index)
        sub = sub.reset_index(drop=True)
    return sub

df_prop_a_train = prep_prophet(df_a.iloc[: int(len(df_a) * 0.8)])
df_prop_a_test  = prep_prophet(df_a.iloc[int(len(df_a) * 0.8):], max_rows=99_999)

df_prop_b_train = prep_prophet(df_b.iloc[: int(len(df_b) * 0.8)])
df_prop_b_test  = prep_prophet(df_b.iloc[int(len(df_b) * 0.8):], max_rows=99_999)

print("Prophet train A:", len(df_prop_a_train), "| test A:", len(df_prop_a_test))
print("Prophet train B:", len(df_prop_b_train), "| test B:", len(df_prop_b_test))

In [ ]:
def build_prophet():
    m = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode="multiplicative",
        changepoint_prior_scale=0.05,
    )
    m.add_regressor("temperatura")
    return m

print("Entrenando Prophet — Periodo A ...")
prophet_a = build_prophet()
prophet_a.fit(df_prop_a_train)

print("Entrenando Prophet — Periodo B ...")
prophet_b = build_prophet()
prophet_b.fit(df_prop_b_train)

print("Listo.")

In [ ]:
pred_prop_a = prophet_a.predict(df_prop_a_test[["ds", "temperatura"]])["yhat"].values
pred_prop_b = prophet_b.predict(df_prop_b_test[["ds", "temperatura"]])["yhat"].values

print("=== Prophet ===")
m_prop_a = metrics(df_prop_a_test["y"].values, pred_prop_a, "Prophet — Periodo A (2016-2019)")
m_prop_b = metrics(df_prop_b_test["y"].values, pred_prop_b, "Prophet — Periodo B (2020-2024)")

### 4.1 Descomposición de componentes (Prophet — Periodo B)

Prophet permite visualizar la tendencia, la estacionalidad anual, semanal y diaria de forma separada, lo cual es valioso para comunicar patrones de consumo a tomadores de decisión.

In [ ]:
future_b = prophet_b.make_future_dataframe(periods=0, freq="h")
temp_all = pd.concat([
    df_prop_b_train[["ds", "temperatura"]],
    df_prop_b_test[["ds", "temperatura"]],
])
future_b = future_b.merge(temp_all, on="ds", how="left").ffill()

forecast_b = prophet_b.predict(future_b)
fig_decomp = prophet_b.plot_components(forecast_b)
plt.suptitle("Descomposición Prophet — Periodo B (2020-2024)", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("../docs/img/prophet_components_b.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Comparación de métricas

In [ ]:
results = pd.DataFrame([
    {"Modelo": "Random Forest", "Periodo": "A (2016-2019)", **m_rf_a},
    {"Modelo": "Random Forest", "Periodo": "B (2020-2024)", **m_rf_b},
    {"Modelo": "Prophet",       "Periodo": "A (2016-2019)", **m_prop_a},
    {"Modelo": "Prophet",       "Periodo": "B (2020-2024)", **m_prop_b},
]).set_index(["Modelo", "Periodo"])

results["MAE"]  = results["MAE"].map("{:.1f}".format)
results["RMSE"] = results["RMSE"].map("{:.1f}".format)
results["MAPE"] = results["MAPE"].map("{:.2f}%".format)

print(results.to_string())
results

## 6. Visualización — Real vs Predicho

Se grafica una semana representativa del conjunto de test de cada periodo para visualizar el ajuste horario de los dos modelos.

In [ ]:
N_PLOT = 24 * 7  # una semana

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

for ax, y_true, pred_rf, label, color in [
    (axes[0], y_test_a.values[:N_PLOT], pred_rf_a[:N_PLOT],
     "Periodo A — semana muestra (2019, test)", "#2563eb"),
    (axes[1], y_test_b.values[:N_PLOT], pred_rf_b[:N_PLOT],
     "Periodo B — semana muestra (2024, test)", "#16a34a"),
]:
    horas = np.arange(len(y_true))
    ax.plot(horas, y_true,    color="black",  linewidth=1.5, label="Real",          alpha=0.85)
    ax.plot(horas, pred_rf,   color=color,    linewidth=1.5, label="RF pred.",      alpha=0.85, linestyle="--")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("Hora (índice)")
    ax.set_ylabel("Demanda (MWh)")
    ax.legend(fontsize=9)
    ax.set_xticks(np.arange(0, N_PLOT + 1, 24))
    ax.set_xticklabels([f"D{i+1}" for i in range(8)], fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../docs/img/real_vs_pred_rf.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.1 Error por hora del día

El error absoluto medio agrupado por hora del día permite identificar si el modelo falla sistemáticamente en horas pico (demanda alta) o en horas valle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, X_test, y_true, pred_rf, title in [
    (axes[0], X_test_a, y_test_a, pred_rf_a, "Error por hora — Periodo A"),
    (axes[1], X_test_b, y_test_b, pred_rf_b, "Error por hora — Periodo B"),
]:
    err_df = pd.DataFrame({
        "hora":    X_test["hora"].values,
        "ae_rf":   np.abs(y_true.values - pred_rf),
    })
    err_hora = err_df.groupby("hora")["ae_rf"].mean()

    ax.bar(err_hora.index, err_hora.values, color="#e07b54", alpha=0.8, label="RF MAE")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Hora CENACE (1-24)")
    ax.set_ylabel("MAE (MWh)")
    ax.set_xticks(range(1, 25))
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("../docs/img/mae_por_hora.png", dpi=150, bbox_inches="tight")
plt.show()